# Apache Spark com Apache Iceberg

Demonstração de operações CRUD (INSERT, UPDATE, DELETE) com **PySpark** e **Apache Iceberg**.

**Cenário:** Sistema de Gestão de Vendas — TechStore  
**Tabelas:** local.vendas.clientes, local.vendas.produtos, local.vendas.pedidos

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("IcebergLocalDevelopment") \
    .config('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1') \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "spark-warehouse/iceberg") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark

## Cenário — TechStore

Mesmo sistema de vendas, agora com **Apache Iceberg** como formato de tabela.

### Diferença principal: Namespaces

No Iceberg as tabelas ficam dentro de **namespaces** (ex: `local.vendas.clientes`),
enquanto no Delta Lake ficam no catálogo padrão (ex: `clientes`).

## DDL — Criação do Namespace e Tabelas

In [ ]:
# Cria namespace (equivalente a schema/database)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.vendas")
spark.sql("SHOW NAMESPACES IN local").show()

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.vendas.clientes (
        id      INT,
        nome    STRING,
        email   STRING,
        cidade  STRING,
        estado  STRING
    )
    USING iceberg
""")
spark.sql("SELECT * FROM local.vendas.clientes").show()

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.vendas.produtos (
        id        INT,
        nome      STRING,
        categoria STRING,
        preco     FLOAT,
        estoque   INT
    )
    USING iceberg
""")
spark.sql("SELECT * FROM local.vendas.produtos").show()

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.vendas.pedidos (
        id           INT,
        cliente_id   INT,
        produto_id   INT,
        quantidade   INT,
        data_pedido  STRING,
        status       STRING
    )
    USING iceberg
""")
spark.sql("SELECT * FROM local.vendas.pedidos").show()

## INSERT — Inserindo Dados

In [ ]:
spark.sql("""
    INSERT INTO local.vendas.clientes VALUES
        (1, 'Ana Silva',       'ana@email.com',      'Sao Paulo',      'SP'),
        (2, 'Carlos Oliveira', 'carlos@email.com',   'Rio de Janeiro', 'RJ'),
        (3, 'Maria Santos',    'maria@email.com',    'Curitiba',       'PR'),
        (4, 'Joao Costa',      'joao@email.com',     'Porto Alegre',   'RS'),
        (5, 'Fernanda Lima',   'fernanda@email.com', 'Belo Horizonte', 'MG')
""")
spark.sql("SELECT * FROM local.vendas.clientes").show()

In [ ]:
spark.sql("""
    INSERT INTO local.vendas.produtos VALUES
        (1, 'Notebook Dell',      'Informatica',  3599.99, 15),
        (2, 'Smartphone Samsung', 'Celulares',    1299.00, 50),
        (3, 'Monitor LG 27',      'Informatica',   899.90, 30),
        (4, 'Teclado Mecanico',   'Perifericos',   349.90, 100),
        (5, 'Mouse Logitech',     'Perifericos',   159.90, 80)
""")
spark.sql("SELECT * FROM local.vendas.produtos").show()

In [ ]:
spark.sql("""
    INSERT INTO local.vendas.pedidos VALUES
        (1, 1, 2, 2, '2024-01-10', 'entregue'),
        (2, 2, 1, 1, '2024-01-12', 'entregue'),
        (3, 3, 3, 1, '2024-01-15', 'em_transporte'),
        (4, 1, 4, 1, '2024-01-20', 'processando'),
        (5, 4, 5, 3, '2024-01-22', 'cancelado')
""")
spark.sql("SELECT * FROM local.vendas.pedidos").show()

## Consulta com JOIN

In [ ]:
spark.sql("""
    SELECT
        p.id         AS pedido_id,
        c.nome       AS cliente,
        pr.nome      AS produto,
        p.quantidade,
        p.status,
        p.data_pedido
    FROM local.vendas.pedidos p
    JOIN local.vendas.clientes c  ON p.cliente_id = c.id
    JOIN local.vendas.produtos pr ON p.produto_id = pr.id
    ORDER BY p.id
""").show(truncate=False)

## UPDATE — Atualizando Dados

In [ ]:
spark.sql("UPDATE local.vendas.pedidos SET status = 'entregue' WHERE id = 3")
spark.sql("SELECT * FROM local.vendas.pedidos WHERE id = 3").show()

In [ ]:
spark.sql("UPDATE local.vendas.produtos SET preco = 1199.00, estoque = 45 WHERE id = 2")
spark.sql("SELECT * FROM local.vendas.produtos WHERE id = 2").show()

## DELETE — Removendo Dados

In [ ]:
spark.sql("DELETE FROM local.vendas.pedidos WHERE status = 'cancelado'")
spark.sql("SELECT * FROM local.vendas.pedidos").show()

## ALTER TABLE — Evolução de Schema

O Iceberg suporta evolução de schema nativamente.

In [ ]:
spark.sql("ALTER TABLE local.vendas.clientes ADD COLUMN telefone STRING")
spark.sql("SELECT * FROM local.vendas.clientes").show()

In [ ]:
spark.sql("UPDATE local.vendas.clientes SET telefone = '(11) 91234-5678' WHERE id = 1")
spark.sql("UPDATE local.vendas.clientes SET telefone = '(21) 99876-5432' WHERE id = 2")
spark.sql("SELECT * FROM local.vendas.clientes").show()

## MERGE — Upsert (Insert or Update)

Insere o registro se não existir, ou atualiza se já existir.

In [ ]:
spark.sql("""
    MERGE INTO local.vendas.clientes AS target
    USING (
        SELECT 6 AS id, 'Pedro Alves' AS nome, 'pedro@email.com' AS email,
               'Fortaleza' AS cidade, 'CE' AS estado, '(85) 98765-4321' AS telefone
    ) AS source
    ON target.id = source.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("SELECT * FROM local.vendas.clientes").show()

## Snapshots — Time Travel no Iceberg

O Iceberg usa **snapshots** para controle de versões. Cada operação DML gera um novo snapshot.

In [ ]:
# Lista todos os snapshots da tabela clientes
spark.sql("SELECT * FROM local.vendas.clientes.snapshots").show(truncate=False)

In [ ]:
snapshots = spark.sql(
    "SELECT snapshot_id, committed_at FROM local.vendas.clientes.snapshots ORDER BY committed_at"
)
snapshots.show(truncate=False)

# Time travel: le estado do primeiro snapshot
primeiro_snapshot = snapshots.collect()[0]['snapshot_id']
print(f"\nClientes no primeiro snapshot (id={primeiro_snapshot}):")
spark.read.option("snapshot-id", primeiro_snapshot).table("local.vendas.clientes").show()

In [ ]:
spark.stop()
print("Sessao Spark encerrada.")